In [22]:
import copy

import matplotlib.pyplot as plt
import numpy as np
import open3d as o3d

# Weekly Project 5 
## Implementation of global registration 
### Task 1
Today, your task is to implement a global registration algorithm.

It should be able to roughly align two point clouds.
Implement the global registration, and then try the following:

1. Can you fit `r1.pcd` and `r2.pcd`?
2. Can you fit `car1.ply` and `car2.ply`?
The corresponding files are in the `global_registration` folder.

*We start by implementing the function to draw the point clouds used in the weekly exercises.*

In [23]:
def draw_registrations(source, target, transformation=None, recolor=False):
    source_temp = copy.deepcopy(source)
    target_temp = copy.deepcopy(target)
    if recolor:
        source_temp.paint_uniform_color([1, 0.706, 0])
        target_temp.paint_uniform_color([0, 0.651, 0.929])
    if transformation is not None:
        source_temp.transform(transformation)
    o3d.visualization.draw_geometries([source_temp, target_temp])

*We load our pointclouds*

In [24]:
source = o3d.io.read_point_cloud("global_registration/r1.pcd")
target = o3d.io.read_point_cloud("global_registration/r2.pcd")

#draw_registrations(source, target, recolor=True)

*And we use another of the functions used in this weeks exercises. This one to first downsample, then find normals and then find the fpfh features:*

In [25]:
def preprocess_point_cloud(pcd, voxel_size):
    print(":: Downsample with a voxel size %.3f." % voxel_size)
    pcd_down = pcd.voxel_down_sample(voxel_size)

    radius_normal = voxel_size * 2
    print(":: Estimate normal with search radius %.3f." % radius_normal)
    pcd_down.estimate_normals(
        o3d.geometry.KDTreeSearchParamHybrid(radius=radius_normal, max_nn=30)
    )

    radius_feature = voxel_size * 5
    print(":: Compute FPFH feature with search radius %.3f." % radius_feature)
    pcd_fpfh = o3d.pipelines.registration.compute_fpfh_feature(
        pcd_down,
        o3d.geometry.KDTreeSearchParamHybrid(
            radius=radius_feature, max_nn=100
        ),
    )
    return pcd_down, pcd_fpfh

Tweaking global registration parameters:
- We tried to tweak the voxel size to increase accuracy.
  - This worked by going from 0.05 to 0.075 removing a bit more detail
  - But trying to lower it to 0.02 made the computation time way too long when using the PointToPoint RANSAC
- We tried both PointToPoint and PointToPlane, but found point to point to work better in this case.

In [26]:
voxel_size = 0.075
source_down, source_fpfh = preprocess_point_cloud(source, voxel_size)
target_down, target_fpfh = preprocess_point_cloud(target, voxel_size)

#draw_registrations(source_down, target_down, recolor=True)

:: Downsample with a voxel size 0.075.
:: Estimate normal with search radius 0.150.
:: Compute FPFH feature with search radius 0.375.
:: Downsample with a voxel size 0.075.
:: Estimate normal with search radius 0.150.
:: Compute FPFH feature with search radius 0.375.


In [27]:
distance_threshold = voxel_size * 1.5

point_to_point = (
    o3d.pipelines.registration.TransformationEstimationPointToPoint(False)
)

result = (
    o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
        source_down,
        target_down,
        source_fpfh,
        target_fpfh,
        True,
        distance_threshold,
        point_to_point
    )
)

print(result)
draw_registrations(source, target, result.transformation,True)

RegistrationResult with fitness=6.843083e-01, inlier_rmse=4.510208e-02, and correspondence_set size of 1474
Access transformation to get result.


## Here is the global registration:
![Global reg](global.png)


## Aligning the cars

In [28]:
# We start by loading the pointclouds:

car_source = o3d.io.read_point_cloud("global_registration/car1.ply")
car_target = o3d.io.read_point_cloud("global_registration/car2.ply")

#draw_registrations(car_source, car_target, recolor=True)

In [29]:
# We use the function to downsample, get normals and features:
voxel_size = 0.07
car_source_down, car_source_fpfh = preprocess_point_cloud(car_source, voxel_size)
car_target_down, car_target_fpfh = preprocess_point_cloud(car_target, voxel_size)

#draw_registrations(car_source_down, car_target_down, recolor=True)

:: Downsample with a voxel size 0.070.
:: Estimate normal with search radius 0.140.
:: Compute FPFH feature with search radius 0.350.
:: Downsample with a voxel size 0.070.
:: Estimate normal with search radius 0.140.
:: Compute FPFH feature with search radius 0.350.


In [30]:
# We can do RANSAC on the features to get a global registration using the point to point:

distance_threshold = voxel_size * 1.5

point_to_point = (
    o3d.pipelines.registration.TransformationEstimationPointToPoint(False)
)

point_to_plane = (
    o3d.pipelines.registration.TransformationEstimationPointToPlane()
)

car_ransac_result = (
    o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
        car_source_down,
        car_target_down,
        car_source_fpfh,
        car_target_fpfh,
        True,
        distance_threshold,
        point_to_point
    )
)

print(car_ransac_result)
#draw_registrations(source, target, car_ransac_result.transformation,True)

RegistrationResult with fitness=9.968136e-01, inlier_rmse=3.547268e-02, and correspondence_set size of 1877
Access transformation to get result.



### Task 2 (Challenge)
Challanges attempt either or both:
- Implement local registration.

- Attempt to reconstruct the car from the images in `car_challange` folder.

You can use the notebooks from Monday as a starting point.

In [31]:
# We start with trying to do local registration on the cars that we have just done global registration on:
# We use ICP with point to plane on the downsampled pointclouds and compare the registration before and after.
threshold = voxel_size * 1.5

trans_init = car_ransac_result.transformation

# Evaluate registration.
print("Initial alignment (global): ")
evaluation = o3d.pipelines.registration.evaluate_registration(
    car_source_down, car_target_down, threshold, trans_init
)
print(evaluation)


print("Alignment after ICP: ")
point_to_plane = (
    o3d.pipelines.registration.TransformationEstimationPointToPlane()
)

result_icp = o3d.pipelines.registration.registration_icp(
    car_source_down, car_target_down, threshold, trans_init, point_to_plane
)

print(result_icp)

Initial alignment (global): 
RegistrationResult with fitness=9.968136e-01, inlier_rmse=3.547268e-02, and correspondence_set size of 1877
Access transformation to get result.
Alignment after ICP: 
RegistrationResult with fitness=9.962825e-01, inlier_rmse=1.928488e-02, and correspondence_set size of 1876
Access transformation to get result.


*So we see how the both the fitness and the rmse went down after the ICP making it harder to judge whether this was actually an improvemnt. We can test it by widening the threshold so the fitness becomes 1 for both transformations. Here We see that it has a lower rmse after the ICP meaning it is a better fit.*

In [32]:
for name, T in [("RANSAC", trans_init), ("ICP", result_icp.transformation)]:
    ev = o3d.pipelines.registration.evaluate_registration(
        car_source_down, car_target_down, voxel_size * 3, T)
    print(name, ev.fitness, ev.inlier_rmse)

RANSAC 1.0 0.036174259347716425
ICP 1.0 0.020913208045093022


In [33]:
#draw_registrations(car_source_down, car_target_down, trans_init, recolor=True)

# Attempting to reconstruct the car

*Just reusing the functions from the exercises:*

In [34]:
def load_pcd(frame_id, camera):
    """
    Load one RGB-D frame and convert it into an Open3D point cloud.
    """
    # Read raw data of frame
    color_raw = o3d.io.read_image(f"car_challange/rgb/{frame_id:07d}.jpg")
    depth_raw = o3d.io.read_image(f"car_challange/depth/{frame_id:07d}.png")
    
    # Get RGBD
    rgbd_image = o3d.geometry.RGBDImage.create_from_color_and_depth(
        color_raw, depth_raw, convert_rgb_to_intensity=True
    )
    
    # Get point cloud
    pcd = o3d.geometry.PointCloud.create_from_rgbd_image(rgbd_image, camera)
    pcd.transform([[1, 0, 0, 0], [0, -1, 0, 0], [0, 0, -1, 0], [0, 0, 0, 1]])
    return pcd

In [ ]:
voxel_size = 0.075
threshold_local = 0.075
threshold_global = voxel_size * 1.5
submap_radius = 1.0  # meters around the predicted frame position
frame_step = 3
n_frames = 1722

normal_param = o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30)
fpfh_param = o3d.geometry.KDTreeSearchParamHybrid(
    radius=voxel_size * 5, max_nn=100
)
point_to_point = o3d.pipelines.registration.TransformationEstimationPointToPoint(
    False
)
point_to_plane = o3d.pipelines.registration.TransformationEstimationPointToPlane()
ransac_criteria = o3d.pipelines.registration.RANSACConvergenceCriteria(
    max_iteration=50000, confidence=0.95
)

camera = o3d.camera.PinholeCameraIntrinsic(
    o3d.camera.PinholeCameraIntrinsicParameters.PrimeSenseDefault
)


def compute_fpfh(pcd):
    return o3d.pipelines.registration.compute_fpfh_feature(pcd, fpfh_param)


def make_submap(merged, center, radius):
    # Crops to keep only points inside a box with width 2*radius.
    sub = o3d.geometry.AxisAlignedBoundingBox(
        min_bound=center - radius, max_bound=center + radius
    )
    return merged.crop(sub)


merged = load_pcd(1, camera)
merged = merged.voxel_down_sample(voxel_size)  # downsample FIRST, then normals
merged.estimate_normals(normal_param)

prev_transform = np.identity(4)

for idx in range(frame_step, n_frames, frame_step):
    new_pcd = load_pcd(idx, camera)
    new_down = new_pcd.voxel_down_sample(voxel_size)
    new_down.estimate_normals(normal_param)

    # --- local submap around predicted frame position ---
    predicted = copy.deepcopy(new_down).transform(prev_transform) # Do a deepcopy to not transform double. 
    # We assume that the new frame will be relatively close to previous frame, therefore we make the submap around here.
    center = (predicted.get_min_bound() + predicted.get_max_bound()) / 2.0 # get the center 3d point
    submap = make_submap(merged, center, submap_radius) # make the submap around the center.
    if len(submap.points) < 100:  # not enough local map yet, fall back
        submap = merged

    # First: try ICP against the submap
    mode = "local"
    icp_res = o3d.pipelines.registration.registration_icp(
        new_down,
        submap,
        threshold_local,
        prev_transform,
        point_to_plane,
    )

    # If bad ICP, try global registration against the submap, then ICP
    if icp_res.fitness < 0.75:
        mode = "global"
        new_down_fpfh = compute_fpfh(new_down)
        submap_fpfh = compute_fpfh(submap)  # small map -> cheap, no caching needed

        ransac = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
            new_down,
            submap,
            new_down_fpfh,
            submap_fpfh,
            True,
            threshold_global,
            point_to_point,
            ransac_n=3,
            checkers=[
                o3d.pipelines.registration.CorrespondenceCheckerBasedOnEdgeLength(
                    0.9
                ),
                o3d.pipelines.registration.CorrespondenceCheckerBasedOnDistance(
                    threshold_global
                ),
            ],
            criteria=ransac_criteria,
        )

        icp_res = o3d.pipelines.registration.registration_icp(
            new_down,
            submap,
            threshold_local,
            ransac.transformation,
            point_to_plane,
        )

    # If the fit is too bad after the global+local skip the frame.
    if icp_res.fitness < 0.65:
        print(f"frame {idx}: rejected, fitness={icp_res.fitness:.3f}")
        continue

    # merge only the downsampled frame, never re-downsample the whole map
    new_down.transform(icp_res.transformation)
    merged += new_down
    prev_transform = icp_res.transformation

    print(
        f"[{mode}] frame {idx:>4}: fitness={icp_res.fitness:.3f}  points in map={len(merged.points)}"
    )

o3d.visualization.draw_geometries([merged])

[global] frame    3: fitness=0.742  points in map=3360
[global] frame    6: fitness=0.749  points in map=5034
[local] frame    9: fitness=0.752  points in map=6706
[local] frame   12: fitness=0.766  points in map=8377
[local] frame   15: fitness=0.762  points in map=10036
[local] frame   18: fitness=0.760  points in map=11700
[local] frame   21: fitness=0.757  points in map=13353
[local] frame   24: fitness=0.752  points in map=15016
[local] frame   27: fitness=0.754  points in map=16730
[local] frame   30: fitness=0.752  points in map=18429
[local] frame   33: fitness=0.760  points in map=20123
[local] frame   36: fitness=0.753  points in map=21789
[local] frame   39: fitness=0.752  points in map=23465
[global] frame   42: fitness=0.745  points in map=25118
[global] frame   45: fitness=0.736  points in map=26755
[global] frame   48: fitness=0.743  points in map=28388
[local] frame   51: fitness=0.767  points in map=30006
[local] frame   54: fitness=0.779  points in map=31630
[local] f

# Car Challenge approaches:

The goal was to merge about 1722 depth frames into a single map with Open3D. Our starting point was the loop seen in the exercise solutions that registered each frame against the growing map with point-to-plane ICP. It worked well for the first ~300 frames, then fitness dropped and drift built up. We tried to reject frames with too low fitness, but frames got rejected and the icp couldnt recover when the distance between the pictures got too big. 

### The baseline

Each new frame was downsampled and registered with ICP against the merged map. After each merge the whole map was voxel-downsampled again. This failed for a few reasons. A single threshold leaves ICP stuck in local minima and one bad frame corrupts everything registered after it.

### What we tried

First we added a global fallback: when local ICP failed, run FPFH features and RANSAC, then refine with ICP. However, we computed features on the whole map, which grew to too many points making it too slow.

We considered splitting the sequence into five submaps, building each one independently and registering the submaps to each other. The drift should stay smaller inside each chunk as we had seen for the first ~300 images. Then only four alignments should be needed at the end. It's a sound approach, but we didn't pursue it.

We also considered an "initial skeleton idea" where we would register about ten evenly spaced frames to each other to make a rough alignment of the car based on global ransac for 10 very different angles of the car. Afterwards, then loop through every frame against that fixed skeleton. This failed however. Our initial frames were roughly 1700/10 = 170 frames apart and barely overlapped making the initial skeleton bad.

## Final approach

The final version that worked best kept the original loop but registers against a smaller local submap instead of the whole thing. For each frame, the previous pose is applied to the new frame to predict where it should land, and a 1 m box around that position is cropped out of the map. If the crop has fewer than 100 points, it falls back to the full map.

ICP runs against that submap first. If fitness is below 0.75, it runs FPFH and RANSAC against the submap, which is small enough to be cheap (therefore avoiding the massive runtimes that was a problem at first), then refines with ICP. Frames with fitness under 0.65 after that are rejected completely as they would only make the pointcloud worse and corrupt later iterations.

Normals are estimated after downsampling, only the downsampled frame is appended to the map, and the map is never re-downsampled (We tried resampling every 300 frames but it did not improve the pointcloud). We also moved to `frame_step = 3` for more overlap between frames, and `voxel_size = 0.075` to keep the cost manageable.

# Final pointcloud:

![Final pointcloud](image1.png)

![Final pointcloud](image2.png)

![Final pointcloud](image3.png)
